# 🚬 흡연 분류 V11 - Pseudo-Labeling (준지도학습)

## 핵심 전략
- ✅ **Pseudo-Labeling**: 확신도 높은 Test 예측을 학습에 활용
- ✅ **V3 설정 유지**: IQR 클리핑, StandardScaler, class_weight='balanced'
- ✅ **확신도 필터링**: ≥0.90 또는 ≤0.10만 선택
- ✅ **가중치 차등**: Train(1.0) > Pseudo(0.5)
- ✅ **V10 피처 엔지니어링 포함** (50+개 피처)

---

## STEP 0: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

print("✅ STEP 0: 환경 설정 완료!")

## STEP 1: 데이터 로드 + 기본 확인

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 60)
print("📊 STEP 1: 데이터 로드 완료")
print("=" * 60)
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\n컬럼명:\n{train.columns.tolist()}")

# 클래스 분포
print(f"\n🎯 클래스 분포:")
print(train['label'].value_counts())
print(f"\n클래스 분포 비율:")
print(train['label'].value_counts(normalize=True))

# 클래스 비율 계산
n_neg = (train['label'] == 0).sum()
n_pos = (train['label'] == 1).sum()
class_ratio = n_neg / n_pos
print(f"\n클래스 비율 (scale_pos_weight): {class_ratio:.2f}")

## STEP 2: 한글 컬럼 매핑 + 피처 엔지니어링

In [ ]:
def map_korean_columns(df):
    """
    한글 컬럼명을 영어로 매핑 (V6~V10에서 검증됨)
    """
    df = df.copy()
    col_map = {}
    
    for col in df.columns:
        c = col.lower()
        if 'id' in c:
            col_map[col] = 'id'
        elif '나이' in col or 'age' in c:
            col_map[col] = 'age'
        elif '키' in col and ('cm' in col or '(' in col):
            col_map[col] = 'height'
        elif '몸무게' in col or '체중' in col:
            col_map[col] = 'weight'
        elif 'bmi' in c:
            col_map[col] = 'bmi'
        elif '시력' in col and '좌' in col:
            col_map[col] = 'eyesight_left'
        elif '시력' in col and '우' in col:
            col_map[col] = 'eyesight_right'
        elif '청력' in col and '좌' in col:
            col_map[col] = 'hearing_left'
        elif '청력' in col and '우' in col:
            col_map[col] = 'hearing_right'
        elif '충치' in col:
            col_map[col] = 'cavity'
        elif '수축기' in col or ('혈압' in col and '수축' in col):
            col_map[col] = 'systolic'
        elif '이완기' in col or ('혈압' in col and '이완' in col):
            col_map[col] = 'diastolic'
        elif '혈당' in col or '공복' in col:
            col_map[col] = 'fasting_blood_sugar'
        elif '중성' in col:
            col_map[col] = 'triglyceride'
        elif '크레아티닌' in col or '크레' in col:
            col_map[col] = 'serum_creatinine'
        elif '콜레스테롤' in col and '고밀도' not in col and '저밀도' not in col:
            col_map[col] = 'cholesterol'
        elif '고밀도' in col:
            col_map[col] = 'hdl'
        elif '저밀도' in col:
            col_map[col] = 'ldl'
        elif '헤모글로빈' in col:
            col_map[col] = 'hemoglobin'
        elif '요단백' in col or ('단백' in col and '지단백' not in col):
            col_map[col] = 'urine_protein'
        elif 'ast' in c:
            col_map[col] = 'ast'
        elif 'alt' in c:
            col_map[col] = 'alt'
        elif '감마' in col or 'gtp' in c or 'γ' in col:
            col_map[col] = 'gtp'
        elif 'label' in c:
            col_map[col] = 'label'
        else:
            col_map[col] = col
    
    return df.rename(columns=col_map)

# 컬럼 매핑 적용
train_mapped = map_korean_columns(train)
test_mapped = map_korean_columns(test)

print("✅ 한글 컬럼 매핑 완료")
print(f"매핑 후 컬럼: {train_mapped.columns.tolist()}")

In [ ]:
def create_features_v11(df):
    """
    V10 스타일 피처 엔지니어링 (50+개 피처 생성)
    """
    df = df.copy()
    cols = df.columns.tolist()
    
    # ============================================
    # 1. 콜레스테롤 관련
    # ============================================
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df['hdl'] / (df['ldl'] + 1)
        df['LDL_HDL_ratio'] = df['ldl'] / (df['hdl'] + 1)
        df['LDL_HDL_diff'] = df['ldl'] - df['hdl']
    
    if 'cholesterol' in cols and 'hdl' in cols:
        df['HDL_Chol_ratio'] = df['hdl'] / (df['cholesterol'] + 1)
        df['NonHDL_Chol'] = df['cholesterol'] - df['hdl']
        df['Atherogenic_idx'] = (df['cholesterol'] - df['hdl']) / (df['hdl'] + 1)
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
    
    if 'triglyceride' in cols and 'cholesterol' in cols:
        df['TG_Chol_ratio'] = df['triglyceride'] / (df['cholesterol'] + 1)
    
    # ============================================
    # 2. 간 기능 (GTP)
    # ============================================
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df['gtp'])
        df['GTP_sqrt'] = np.sqrt(df['gtp'])
        df['GTP_sq'] = df['gtp'] ** 2
        df['GTP_high'] = (df['gtp'] > 50).astype(int)
    
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df['ast'] / (df['alt'] + 1)
        df['Liver_sum'] = df['ast'] + df['alt']
        df['Liver_diff'] = abs(df['ast'] - df['alt'])
    
    if 'gtp' in cols and 'ast' in cols:
        df['GTP_AST_ratio'] = df['gtp'] / (df['ast'] + 1)
    
    # ============================================
    # 3. 헤모글로빈 (흡연자 높음)
    # ============================================
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df['hemoglobin'])
        df['Hemo_high'] = (df['hemoglobin'] > 15).astype(int)
        df['Hemo_vhigh'] = (df['hemoglobin'] > 16).astype(int)
    
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df['hemoglobin'] * df['gtp']
    
    if 'hemoglobin' in cols and 'triglyceride' in cols:
        df['Hemo_x_TG'] = df['hemoglobin'] * df['triglyceride']
    
    # ============================================
    # 4. 혈압 관련
    # ============================================
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df['systolic'] - df['diastolic']
        df['MAP'] = df['diastolic'] + (df['systolic'] - df['diastolic']) / 3
        df['BP_ratio'] = df['systolic'] / (df['diastolic'] + 1)
        df['BP_high'] = ((df['systolic'] > 130) | (df['diastolic'] > 85)).astype(int)
    
    # ============================================
    # 5. 체형 관련
    # ============================================
    if 'height' in cols and 'weight' in cols:
        height_m = df['height'] / 100
        df['BMI_calc'] = df['weight'] / (height_m ** 2 + 0.01)
        df['BMI_sq'] = df['BMI_calc'] ** 2
    
    if 'bmi' in cols:
        df['BMI_group'] = pd.cut(df['bmi'], bins=[0, 18.5, 23, 25, 100], labels=[0, 1, 2, 3]).astype(float)
    
    # ============================================
    # 6. 시력
    # ============================================
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df[eye_cols].mean(axis=1)
        df['Eyesight_diff'] = abs(df[eye_cols[0]] - df[eye_cols[1]])
        df['Eyesight_min'] = df[eye_cols].min(axis=1)
    
    # ============================================
    # 7. 나이 관련
    # ============================================
    if 'age' in cols:
        df['Age_sq'] = df['age'] ** 2
        df['Age_group'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
        
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = df['age'] * df['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = df['age'] * df['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = df['age'] * df['triglyceride']
        if 'systolic' in cols:
            df['Age_x_SBP'] = df['age'] * df['systolic']
        if 'cholesterol' in cols:
            df['Age_x_Chol'] = df['age'] * df['cholesterol']
    
    # ============================================
    # 8. 혈당 관련
    # ============================================
    if 'fasting_blood_sugar' in cols:
        df['FBS_log'] = np.log1p(df['fasting_blood_sugar'])
        df['FBS_high'] = (df['fasting_blood_sugar'] > 100).astype(int)
        df['FBS_diabetes'] = (df['fasting_blood_sugar'] > 126).astype(int)
    
    # ============================================
    # 9. 중성지방
    # ============================================
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df['triglyceride'])
        df['TG_sq'] = df['triglyceride'] ** 2
        df['TG_high'] = (df['triglyceride'] > 150).astype(int)
    
    # ============================================
    # 10. 크레아티닌
    # ============================================
    if 'serum_creatinine' in cols:
        df['Creat_log'] = np.log1p(df['serum_creatinine'])
        df['Creat_high'] = (df['serum_creatinine'] > 1.2).astype(int)
    
    # ============================================
    # 11. 종합 건강 점수
    # ============================================
    health_cols = [c for c in ['systolic', 'diastolic', 'hemoglobin', 'triglyceride', 
                               'cholesterol', 'hdl', 'ldl', 'gtp'] if c in cols]
    if len(health_cols) >= 3:
        df['Health_mean'] = df[health_cols].mean(axis=1)
        df['Health_std'] = df[health_cols].std(axis=1)
        df['Health_max'] = df[health_cols].max(axis=1)
        df['Health_min'] = df[health_cols].min(axis=1)
        df['Health_range'] = df['Health_max'] - df['Health_min']
    
    # ============================================
    # 12. 흡연 위험 점수
    # ============================================
    risk_score = 0
    if 'Hemo_high' in df.columns:
        risk_score = risk_score + df['Hemo_high']
    if 'GTP_high' in df.columns:
        risk_score = risk_score + df['GTP_high']
    if 'TG_high' in df.columns:
        risk_score = risk_score + df['TG_high']
    if 'BP_high' in df.columns:
        risk_score = risk_score + df['BP_high']
    df['Smoking_risk_score'] = risk_score
    
    # 결측치/무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 피처 엔지니어링 적용
print("\n🔧 STEP 2: 피처 엔지니어링 적용 중...")

# ID 제거
train_df = train_mapped.drop(['id'], axis=1, errors='ignore')
test_df = test_mapped.drop(['id'], axis=1, errors='ignore')

original_cols = len(train_df.columns) - 1  # label 제외

train_fe = create_features_v11(train_df)
test_fe = create_features_v11(test_df)

print(f"\n✅ STEP 2: 피처 엔지니어링 완료!")
print(f"   원본: {original_cols}개 → 새로운: {train_fe.shape[1]-1}개")
print(f"   추가된 피처: {train_fe.shape[1]-1-original_cols}개")

## STEP 3: 전처리 (이상치 + 스케일링)

In [ ]:
# X, y 분리
X = train_fe.drop('label', axis=1)
y = train_fe['label']
X_test = test_fe.drop('label', axis=1, errors='ignore')

# 컬럼 순서 맞추기
X_test = X_test[X.columns]

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

In [ ]:
# IQR 이상치 클리핑 (V3에서 효과적)
def clip_outliers_iqr(df, multiplier=3.0):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df[col] = df[col].clip(lower=lower, upper=upper)
    
    return df

X_clipped = clip_outliers_iqr(X)
X_test_clipped = clip_outliers_iqr(X_test)

print("✅ IQR 이상치 클리핑 완료 (multiplier=3.0)")

In [ ]:
# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clipped)
X_test_scaled = scaler.transform(X_test_clipped)

print(f"✅ STEP 3: 전처리 완료!")
print(f"   X_scaled: {X_scaled.shape}")
print(f"   X_test_scaled: {X_test_scaled.shape}")

## STEP 4: 1단계 모델 튜닝 (V3 파라미터 기반)

In [ ]:
print("=" * 60)
print("🔧 STEP 4: 1단계 모델 튜닝 (Accuracy 기준)")
print("=" * 60)

In [ ]:
# [1/3] XGBoost 튜닝
print("\n🔄 [1/3] XGBoost 튜닝 중...")

xgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5],
    'scale_pos_weight': [1, class_ratio]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
    xgb_params, n_iter=80, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_scaled, y)
print(f"\n✅ XGBoost 최적 점수: {xgb_search.best_score_:.5f}")

In [ ]:
# [2/3] LightGBM 튜닝
print("\n🔄 [2/3] LightGBM 튜닝 중...")

lgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'class_weight': ['balanced', None]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=80, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_scaled, y)
print(f"\n✅ LightGBM 최적 점수: {lgb_search.best_score_:.5f}")

In [ ]:
# [3/3] CatBoost 튜닝
print("\n🔄 [3/3] CatBoost 튜닝 중...")

cat_params = {
    'iterations': [300, 500, 700, 1000],
    'depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7],
    'auto_class_weights': ['Balanced', None]
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_scaled, y)
print(f"\n✅ CatBoost 최적 점수: {cat_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 60)
print("📊 1단계 튜닝 결과 요약")
print("=" * 60)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")
print("\n✅ 1단계 모델 튜닝 완료!")

## STEP 5: Pseudo-Labeling (핵심!)

In [ ]:
print("\n" + "=" * 60)
print("🎯 STEP 5: Pseudo-Labeling")
print("=" * 60)

# 1단계 모델로 Test 예측
pred_xgb = xgb_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]
pred_lgb = lgb_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]
pred_cat = cat_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]

# 가중 평균 앙상블 (V3 스타일)
W_XGB = 0.40
W_LGB = 0.35
W_CAT = 0.25

ensemble_pred = W_XGB * pred_xgb + W_LGB * pred_lgb + W_CAT * pred_cat

print(f"\n📊 1단계 앙상블 예측 분포:")
print(f"   Min: {ensemble_pred.min():.4f}")
print(f"   Max: {ensemble_pred.max():.4f}")
print(f"   Mean: {ensemble_pred.mean():.4f}")

In [ ]:
# 확신도 필터링 (핵심!)
THRESHOLD_HIGH = 0.90  # 흡연자 확신
THRESHOLD_LOW = 0.10   # 비흡연자 확신

# 확신도 높은 샘플만 선택
high_conf_mask = (ensemble_pred >= THRESHOLD_HIGH) | (ensemble_pred <= THRESHOLD_LOW)

X_pseudo = X_test_scaled[high_conf_mask]
y_pseudo = (ensemble_pred[high_conf_mask] >= 0.5).astype(int)

print(f"\n📊 Pseudo 샘플 통계:")
print(f"   전체 Test: {len(X_test_scaled)}개")
print(f"   Pseudo 선택: {len(X_pseudo)}개 ({len(X_pseudo)/len(X_test_scaled)*100:.1f}%)")
print(f"   ├─ 비흡연 확신 (≤{THRESHOLD_LOW}): {(ensemble_pred <= THRESHOLD_LOW).sum()}개")
print(f"   └─ 흡연 확신 (≥{THRESHOLD_HIGH}): {(ensemble_pred >= THRESHOLD_HIGH).sum()}개")
print(f"\n   Pseudo 클래스 분포:")
print(f"   ├─ 비흡연(0): {(y_pseudo==0).sum()}개 ({(y_pseudo==0).sum()/len(y_pseudo)*100:.1f}%)")
print(f"   └─ 흡연(1): {(y_pseudo==1).sum()}개 ({(y_pseudo==1).sum()/len(y_pseudo)*100:.1f}%)")

In [ ]:
# Train + Pseudo 결합
X_combined = np.vstack([X_scaled, X_pseudo])
y_combined = np.concatenate([y.values, y_pseudo])

# 가중치 설정 (Train > Pseudo)
PSEUDO_WEIGHT = 0.5  # Pseudo 샘플 가중치

sample_weight = np.concatenate([
    np.ones(len(X_scaled)),              # Train: 가중치 1.0
    np.ones(len(X_pseudo)) * PSEUDO_WEIGHT  # Pseudo: 가중치 0.5
])

print(f"\n📊 최종 학습 데이터:")
print(f"   Train: {len(X_scaled)}개 (가중치 1.0)")
print(f"   Pseudo: {len(X_pseudo)}개 (가중치 {PSEUDO_WEIGHT})")
print(f"   합계: {len(X_combined)}개 (+{len(X_pseudo)/len(X_scaled)*100:.1f}%)")
print(f"\n   결합 후 클래스 분포:")
print(f"   ├─ 비흡연(0): {(y_combined==0).sum()}개 ({(y_combined==0).sum()/len(y_combined)*100:.1f}%)")
print(f"   └─ 흡연(1): {(y_combined==1).sum()}개 ({(y_combined==1).sum()/len(y_combined)*100:.1f}%)")

## STEP 6: 2단계 모델 재학습 (Train + Pseudo)

In [ ]:
print("\n" + "=" * 60)
print("🔄 STEP 6: 2단계 모델 재학습 (Train + Pseudo)")
print("=" * 60)

# 최적 파라미터로 재학습
print("\n[1/3] XGBoost 재학습...")
final_xgb = XGBClassifier(
    **xgb_search.best_params_, 
    random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'
)
final_xgb.fit(X_combined, y_combined, sample_weight=sample_weight)
print("   ✅ 완료")

print("\n[2/3] LightGBM 재학습...")
final_lgb = LGBMClassifier(
    **lgb_search.best_params_, 
    random_state=42, verbose=-1
)
final_lgb.fit(X_combined, y_combined, sample_weight=sample_weight)
print("   ✅ 완료")

print("\n[3/3] CatBoost 재학습...")
# CatBoost는 sample_weight 파라미터명이 다름
final_cat_params = cat_search.best_params_.copy()
# iterations -> n_estimators 변환 필요 없음 (CatBoost는 iterations 사용)
final_cat = CatBoostClassifier(
    **final_cat_params,
    random_state=42, verbose=0
)
# CatBoost Pool 사용 (sample_weight 지원)
from catboost import Pool
train_pool = Pool(X_combined, y_combined, weight=sample_weight)
final_cat.fit(train_pool)
print("   ✅ 완료")

print("\n✅ 2단계 재학습 완료!")

In [ ]:
# 최종 Test 예측
final_pred_xgb = final_xgb.predict_proba(X_test_scaled)[:, 1]
final_pred_lgb = final_lgb.predict_proba(X_test_scaled)[:, 1]
final_pred_cat = final_cat.predict_proba(X_test_scaled)[:, 1]

# 최종 앙상블
final_ensemble = W_XGB * final_pred_xgb + W_LGB * final_pred_lgb + W_CAT * final_pred_cat

print("\n📊 2단계 앙상블 예측 분포:")
print(f"   Min: {final_ensemble.min():.4f}")
print(f"   Max: {final_ensemble.max():.4f}")
print(f"   Mean: {final_ensemble.mean():.4f}")

## STEP 7: OOF 기반 최적 임계값 탐색

In [ ]:
print("\n" + "=" * 60)
print("🔍 STEP 7: OOF 기반 최적 임계값 탐색")
print("=" * 60)

# Train 데이터에 대한 OOF 예측 (Combined 데이터 중 원본 Train 부분만)
# 5-Fold OOF로 원본 Train에 대한 예측 생성

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X_scaled))
oof_lgb = np.zeros(len(X_scaled))
oof_cat = np.zeros(len(X_scaled))

print("\n OOF 예측 생성 중...")

for fold, (tr_idx, va_idx) in enumerate(kfold.split(X_scaled, y)):
    print(f"   Fold {fold+1}/5...", end=" ")
    
    X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
    # XGBoost
    xgb_m = XGBClassifier(**xgb_search.best_params_, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
    xgb_m.fit(X_tr, y_tr)
    oof_xgb[va_idx] = xgb_m.predict_proba(X_va)[:, 1]
    
    # LightGBM
    lgb_m = LGBMClassifier(**lgb_search.best_params_, random_state=42, verbose=-1)
    lgb_m.fit(X_tr, y_tr)
    oof_lgb[va_idx] = lgb_m.predict_proba(X_va)[:, 1]
    
    # CatBoost
    cat_m = CatBoostClassifier(**cat_search.best_params_, random_state=42, verbose=0)
    cat_m.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_m.predict_proba(X_va)[:, 1]
    
    print("✓")

# OOF 앙상블
oof_ensemble = W_XGB * oof_xgb + W_LGB * oof_lgb + W_CAT * oof_cat

print("\n✅ OOF 예측 생성 완료!")

In [ ]:
# 최적 임계값 탐색 (0.001 단위)
best_threshold = 0.5
best_acc = 0
best_f1 = 0
results = []

for threshold in np.arange(0.35, 0.65, 0.001):
    pred = (oof_ensemble >= threshold).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': round(threshold, 3), 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        best_threshold = round(threshold, 3)

results_df = pd.DataFrame(results)
print("\n상위 15개 임계값:")
print(results_df.nlargest(15, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_threshold:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"   OOF F1-Score: {best_f1:.5f}")

In [ ]:
# 혼동 행렬
val_pred_final = (oof_ensemble >= best_threshold).astype(int)

print("\n📊 OOF 혼동 행렬:")
print(confusion_matrix(y, val_pred_final))
print("\n📊 Classification Report:")
print(classification_report(y, val_pred_final, target_names=['비흡연(0)', '흡연(1)']))

## STEP 8: 제출 파일 생성 (5개)

In [ ]:
print("\n" + "=" * 60)
print("📁 STEP 8: 제출 파일 생성")
print("=" * 60)

# 최적 임계값 ± 0.02, ± 0.04
thresholds = [
    round(best_threshold - 0.04, 3),
    round(best_threshold - 0.02, 3),
    round(best_threshold, 3),
    round(best_threshold + 0.02, 3),
    round(best_threshold + 0.04, 3)
]

file_paths = []

for th in thresholds:
    pred = (final_ensemble >= th).astype(int)
    oof_pred = (oof_ensemble >= th).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    th_str = str(int(th * 1000)).zfill(3)
    filename = f'submission_v11_pseudo_t{th_str}.csv'
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    n_smoking = (pred == 1).sum()
    pct = n_smoking / len(pred) * 100
    
    marker = "⭐" if th == best_threshold else "  "
    print(f"\n{marker} {filename}")
    print(f"   임계값: {th:.3f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   예측: 비흡연={len(pred)-n_smoking} ({100-pct:.1f}%), 흡연={n_smoking} ({pct:.1f}%)")

print(f"\n✅ {len(thresholds)}개 제출 파일 생성 완료!")

In [ ]:
# 검증
print("\n🔍 제출 파일 검증:")
for fp in file_paths:
    df = pd.read_csv(fp)
    fn = fp.split('/')[-1]
    valid = df['label'].dtype in ['int64', 'int32'] and set(df['label'].unique()).issubset({0, 1})
    print(f"   {'✅' if valid else '❌'} {fn}: {df.shape}, dtype={df['label'].dtype}")

## STEP 9: 다운로드 + 요약

In [ ]:
from google.colab import files

# 최적 임계값 파일 다운로드
best_file = result_path + f'submission_v11_pseudo_t{str(int(best_threshold*1000)).zfill(3)}.csv'
files.download(best_file)

print("\n" + "=" * 60)
print("🎉 V11 Pseudo-Labeling 완료!")
print("=" * 60)
print(f"\n📊 핵심 통계:")
print(f"   모델: XGBoost + LightGBM + CatBoost")
print(f"   가중치: XGB={W_XGB}, LGB={W_LGB}, CAT={W_CAT}")
print(f"   피처 수: {X_scaled.shape[1]}개")
print(f"\n📊 Pseudo-Labeling 결과:")
print(f"   원본 Train: {len(X_scaled)}개")
print(f"   Pseudo 추가: {len(X_pseudo)}개 (+{len(X_pseudo)/len(X_scaled)*100:.1f}%)")
print(f"   확신도 필터: ≤{THRESHOLD_LOW} 또는 ≥{THRESHOLD_HIGH}")
print(f"   Pseudo 가중치: {PSEUDO_WEIGHT}")
print(f"\n📊 최적 임계값: {best_threshold:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n📁 생성된 파일:")
for fp in file_paths:
    fn = fp.split('/')[-1]
    marker = "👉" if f't{str(int(best_threshold*1000)).zfill(3)}' in fn else "  "
    print(f"   {marker} {fn}")
print(f"\n🎯 제출 전략:")
print(f"   1. submission_v11_pseudo_t{str(int(best_threshold*1000)).zfill(3)}.csv 먼저 제출")
print(f"   2. 점수 확인 후 다른 임계값 파일 시도")

In [ ]:
# 다른 파일도 다운로드
print("\n📥 추가 파일 다운로드:")
for fp in file_paths:
    if fp != best_file:
        files.download(fp)
        print(f"   ✅ {fp.split('/')[-1]}")